# Textual description about what is depicted

In [ ]:
import sys, os
print(sys.executable)

/usr/bin/python3


In [ ]:
# %pip install accelerate
# % pip install "transformers>=4.57.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
base_dir = "/content/drive/MyDrive/RA"
image_paths = ["data/FFF_hun/2019-02-26_08-01-54_UTC.jpg","data/FFF_hun/2019-03-02_10-27-43_UTC.jpg", "data/FFF_hun/big_face.jpg","data/FFF_hun/group_marching.jpg","data/FFF_hun/tilted_face.jpg","data/FFF_hun/tilted_up_face.jpg"]
for i,image_file in enumerate(image_paths):
    image_paths[i] = os.path.join(base_dir,image_file)

# VLLM

accelerate
transformers>=4.57.0

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

In [ ]:
# Load LLaVA
LLAVA_MODEL = "llava-hf/llava-1.5-7b-hf"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else "cpu")

In [ ]:
def load_llava_model(llava_model=LLAVA_MODEL):
    """
    Load a pre-trained LLaVA model and processor.

    Args:
        llava_model (str): Hugging Face model identifier.

    Returns:
        tuple: Loaded LLaVA model and processor.
    """
    print("Loading LLaVA model...")

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = LlavaForConditionalGeneration.from_pretrained(
        llava_model,
        torch_dtype=dtype,
        device_map="auto"
    )
    processor = AutoProcessor.from_pretrained(llava_model)
    return model, processor

def describe_image(image, model, processor):
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {
                    "type": "text",
                    "text": (
                              "Describe this image. "
                              "Describe only what is visually observable, "
                              "including people, objects, text, symbols, "
                              "and the overall scene. "
                              "Do not discuss its purpose, meaning, narrative, "
                              "or make assumptions about what the image communicates."
                    )
                }
            ],
        }
    ]
    prompt = processor.apply_chat_template(conversation,add_generation_prompt=True,tokenize=False)
    inputs = processor(images=image,text=prompt,return_tensors="pt")
    inputs = {
        key: value.to(model.device) if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }
    generate_ids = model.generate(**inputs,max_new_tokens=100,do_sample=False)
    input_length = inputs["input_ids"].shape[1]
    generated_ids = generate_ids[:, input_length:]
    result = processor.batch_decode(generated_ids,skip_special_tokens=True)[0].strip()
    return result

# BLIP

In [ ]:
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import os

# Load the pre-trained BLIP model and processor
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 7063.91it/s]


In [ ]:
# Open an image file
base_dir = "C:/Users/lchtu/OneDrive/Desktop/Persoonlijk/werk/RA InfoLAB/Data Transformation/visual-data-transformations/"
image_path = "data/FFF_hun/2019-02-26_08-01-54_UTC.jpg"
image = Image.open(os.path.join(base_dir,image_path))

# Preprocess the image and prepare inputs for the model
text = "Describe this image in detail, including the people, their actions, objects, and setting."
inputs = processor(images=image, text=text, return_tensors="pt")

# Generate caption
caption = model.generate(**inputs)

# Decode the generated caption
caption_text = processor.decode(caption[0], skip_special_tokens=True)

print("Generated Caption:", caption_text)

c:\Users\lchtu\OneDrive\Desktop\Persoonlijk\werk\RA InfoLAB\Data Transformation\visual-data-transformations\.vtransenv\Lib\site-packages\transformers\generation\utils.py:1638: UserWarning: Using the model-agnostic default `max_length` (=39) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Generated Caption: describe this image in detail, including the people, their actions, objects, and setting., the people, their actions, and the people


# Example Usage

In [ ]:
model, processor = load_llava_model(LLAVA_MODEL)

Loading LLaVA model...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

In [ ]:
image = Image.open(image_paths[0]).convert("RGB")
description = describe_image(image,  model, processor)
print(description)
# Image.show(image)

The image features a group of people standing outside a building, holding up signs. There are at least six people in the scene, with some standing closer to the building and others further away. The signs they are holding are visible, with one sign being larger and more prominent than the others.

The people in the scene are wearing winter clothing, indicating that the weather is cold. The building they are standing in front of appears to be a large, possibly historical structure.
